# Estimativa SWIR via GEE — smileGradientTreeBoost

Treina e aplica **Gradient Tree Boosting** diretamente no GEE usando os pontos amostrados do asset.

| Etapa | Descrição |
|---|---|
| Carregar pontos | Asset `projects/mapbiomas-arida/mine/points` |
| Grid search | 6 combinações de params → escolhe por RMSE de validação |
| Modelos finais | 1 classifier por banda SWIR treinado no conjunto completo |
| Aplicar | Classifica o mosaico final → imagem com B04_est–B09_est |
| Exportar | Asset `projects/mapbiomas-arida/mine/swir_estimated` |

> **Mapas interativos** requerem `geemap` (`pip install geemap`). Se não instalado, as células de mapa são opcionais.

In [ ]:
# Dependências
# !pip install earthengine-api geemap pandas matplotlib seaborn

import ee
import json
import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
log = logging.getLogger(__name__)

try:
    import geemap
    HAS_GEEMAP = True
except ImportError:
    HAS_GEEMAP = False
    print('geemap nao instalado — celulas de mapa serao ignoradas')

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
print('Imports OK')

In [ ]:
# ── Inicialização GEE ───────────────────────────────────────────────────────
projAccount = 'mapbiomas-caatinga-cloud02'
try:
    ee.Initialize(project=projAccount)
    log.info('Earth Engine inicializado.')
except Exception as e:
    log.error(f'Erro: {e}')
    raise

## 1. Parâmetros

Ajuste `USE_SAVED_PARAMS = True` para pular o grid search e usar `models/gee_best_params.json` de uma execução anterior.

In [ ]:
ASSET_POINTS  = 'projects/mapbiomas-arida/mine/points'
ASSET_MOSAIC  = 'projects/mapbiomas-arida/MOSAICO_FINAL_ASTER_IRECE'
ASSET_OUTPUT  = 'projects/mapbiomas-arida/mine/swir_estimated'
MODELS_DIR    = Path('models')

QUALITY_MIN   = 5000   # limiar INT16 (≈ quality float 0.5)
VAL_FRAC      = 0.2    # fração do conjunto de validação
SEED          = 42

USE_SAVED_PARAMS = False  # True → lê gee_best_params.json (pula grid search)

FEATURES = ['B01', 'B02', 'B3N', 'B10', 'B11', 'B12', 'B13', 'B14']
TARGETS  = ['B04', 'B05', 'B06', 'B07', 'B08', 'B09']

# Grid de hiperparâmetros do smileGradientTreeBoost
# numberOfTrees ↔ n_estimators  |  shrinkage ↔ learning_rate
# samplingRate  ↔ subsample      |  maxNodes  ↔ ~2^max_depth
PARAM_GRID = [
    {'numberOfTrees': 100, 'shrinkage': 0.10, 'samplingRate': 0.7, 'maxNodes': 64},
    {'numberOfTrees': 200, 'shrinkage': 0.10, 'samplingRate': 0.8, 'maxNodes': 128},
    {'numberOfTrees': 300, 'shrinkage': 0.05, 'samplingRate': 0.8, 'maxNodes': 128},
    {'numberOfTrees': 300, 'shrinkage': 0.10, 'samplingRate': 0.8, 'maxNodes': 128},
    {'numberOfTrees': 500, 'shrinkage': 0.05, 'samplingRate': 0.8, 'maxNodes': 256},
    {'numberOfTrees': 500, 'shrinkage': 0.10, 'samplingRate': 0.9, 'maxNodes': 256},
]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'ASSET_POINTS : {ASSET_POINTS}')
print(f'ASSET_OUTPUT : {ASSET_OUTPUT}')
print(f'Grid search  : {len(PARAM_GRID)} combinacoes x {len(TARGETS)} bandas')

## 2. Carregar Pontos do Asset GEE

In [ ]:
def carregar_pontos():
    fc_list = ee.data.listAssets({'parent': ASSET_POINTS})['assets']
    print(f'Assets de pontos encontrados: {len(fc_list)}')
    for a in fc_list:
        print(f'  {a["id"].split("/")[-1]}')

    fcs = [ee.FeatureCollection(a['id']) for a in fc_list]
    fc  = ee.FeatureCollection(fcs).flatten()
    fc  = fc.filter(ee.Filter.gte('quality', QUALITY_MIN))

    n = fc.size().getInfo()
    print(f'\nTotal de pontos (quality >= {QUALITY_MIN}): {n:,}')
    return fc

fc_all = carregar_pontos()

In [ ]:
# ── Mapa interativo com os pontos e o mosaico (requer geemap) ──────────────
if HAS_GEEMAP:
    mapa = geemap.Map()
    mapa.centerObject(ee.FeatureCollection(fc_all), zoom=8)

    mosaico = ee.Image(ASSET_MOSAIC)
    mapa.addLayer(
        mosaico.divide(10000).float(),
        {'bands': ['B3N', 'B02', 'B01'], 'min': 0.05, 'max': 0.35, 'gamma': 1.4},
        'RGB Natural'
    )
    mapa.addLayer(
        mosaico.divide(10000).float(),
        {'bands': ['B04', 'B06', 'B3N'], 'min': 0.1, 'max': 0.5, 'gamma': 1.2},
        'SWIR Falsa-cor'
    )
    mapa.addLayer(
        fc_all.style(color='yellow', pointSize=2),
        {},
        'Pontos amostrados'
    )
    display(mapa)
else:
    print('geemap nao disponivel — instale com: pip install geemap')

In [ ]:
# ── Estatísticas dos pontos (amostra do lado cliente) ─────────────────────
fc_sample = fc_all.limit(500)
df_sample = pd.DataFrame(
    [f['properties'] for f in fc_sample.getInfo()['features']]
)
cols_vis  = [c for c in FEATURES + TARGETS + ['quality'] if c in df_sample.columns]
df_sample[cols_vis].describe().round(2).style.background_gradient(cmap='Blues', axis=0)

In [ ]:
# ── Distribuição das bandas na amostra (500 pontos) ───────────────────────
def scale_band(col, df):
    # INT16 → reflectancia (VNIR/SWIR) ou temperatura (TIR)
    fator = 1e-4 if col in ['B01','B02','B3N','B04','B05','B06','B07','B08','B09'] else 0.1
    return df[col] * fator if col in df.columns else None

bandas_plot = [c for c in TARGETS if c in df_sample.columns]
if bandas_plot:
    fig, axes = plt.subplots(1, len(bandas_plot), figsize=(14, 3))
    for ax, banda in zip(axes, bandas_plot):
        vals = scale_band(banda, df_sample)
        if vals is not None:
            ax.hist(vals, bins=30, color='#4C72B0', edgecolor='white', linewidth=0.3)
        ax.set_title(banda, fontsize=10)
        ax.set_xlabel('Reflectancia')
    plt.suptitle('Distribuicao SWIR — Amostra de 500 pontos', y=1.03)
    plt.tight_layout()
    plt.show()

## 3. Split Treino / Validação

`randomColumn` no GEE garante divisão reprodutível com `seed=42`.

In [ ]:
def split_train_val(fc, val_frac=VAL_FRAC):
    fc_r   = fc.randomColumn('_rand', seed=SEED)
    fc_tr  = fc_r.filter(ee.Filter.gt('_rand', val_frac))
    fc_val = fc_r.filter(ee.Filter.lte('_rand', val_frac))
    return fc_tr, fc_val

fc_tr, fc_val = split_train_val(fc_all)

n_tr  = fc_tr.size().getInfo()
n_val = fc_val.size().getInfo()
print(f'Treino    : {n_tr:,} pontos  ({100*(1-VAL_FRAC):.0f}%)')
print(f'Validacao : {n_val:,} pontos  ({100*VAL_FRAC:.0f}%)')

## 4. Grid Search de Hiperparâmetros

Cada combinação do `PARAM_GRID` é treinada no conjunto de treino e avaliada pelo RMSE no conjunto de validação.

> **Tempo estimado:** ~3–5 min por banda × 6 bandas ≈ 20–30 min total.

In [ ]:
def treinar_gbt(fc_tr, target, params):
    return (
        ee.Classifier.smileGradientTreeBoost(
            numberOfTrees=params['numberOfTrees'],
            shrinkage=params['shrinkage'],
            samplingRate=params['samplingRate'],
            maxNodes=params.get('maxNodes'),
            seed=SEED,
        )
        .setOutputMode('REGRESSION')
        .train(
            features=fc_tr,
            classProperty=target,
            inputProperties=FEATURES,
        )
    )

def calcular_rmse(fc_val, clf, target):
    fc_pred = fc_val.classify(clf, '_pred')
    fc_err  = fc_pred.map(
        lambda f: f.set(
            '_err_sq',
            f.getNumber('_pred').subtract(f.getNumber(target)).pow(2)
        )
    )
    mse = (
        fc_err
        .reduceColumns(ee.Reducer.mean(), ['_err_sq'])
        .get('mean')
        .getInfo()
    )
    return float(mse) ** 0.5

print('Funcoes treinar_gbt e calcular_rmse definidas')

In [ ]:
gee_best_params_path = MODELS_DIR / 'gee_best_params.json'

if USE_SAVED_PARAMS and gee_best_params_path.exists():
    with open(gee_best_params_path) as f:
        saved = json.load(f)
    melhores_params = {b: saved[b]['params'] for b in TARGETS}
    gee_metrics     = {b: {'val_rmse': saved[b]['val_rmse'], 'params': saved[b]['params']} for b in TARGETS}
    print('Params carregados de gee_best_params.json (grid search ignorado)')

else:
    melhores_params = {}
    gee_metrics     = {}
    resultados_grid = []  # para visualizacao

    for banda in TARGETS:
        print(f'\n--- Grid search: {banda} ---')
        melhor_rmse, melhor_params = float('inf'), PARAM_GRID[0]

        for i, params in enumerate(PARAM_GRID, 1):
            try:
                clf  = treinar_gbt(fc_tr, banda, params)
                rmse = calcular_rmse(fc_val, clf, banda)
                print(f'  [{i}/{len(PARAM_GRID)}] nTrees={params["numberOfTrees"]:3d}  '
                      f'lr={params["shrinkage"]:.2f}  maxNodes={params["maxNodes"]:3d}  '
                      f'RMSE={rmse:.4f}')
                resultados_grid.append({'banda': banda, **params, 'val_rmse': rmse})
                if rmse < melhor_rmse:
                    melhor_rmse, melhor_params = rmse, params
            except Exception as exc:
                print(f'  [{i}/{len(PARAM_GRID)}] FALHOU: {exc}')
            time.sleep(0.3)

        melhores_params[banda] = melhor_params
        gee_metrics[banda]     = {'val_rmse': melhor_rmse, 'params': melhor_params}
        print(f'  >> Melhor: nTrees={melhor_params["numberOfTrees"]}  '
              f'lr={melhor_params["shrinkage"]}  RMSE={melhor_rmse:.4f}')

    # Salvar resultados
    with open(gee_best_params_path, 'w') as f:
        json.dump(gee_metrics, f, indent=2)
    print(f'\nParams salvos -> {gee_best_params_path}')

In [ ]:
# ── Tabela de resultados do grid search ───────────────────────────────────
df_best = pd.DataFrame([
    {'Banda': b,
     'nTrees': gee_metrics[b]['params']['numberOfTrees'],
     'shrinkage': gee_metrics[b]['params']['shrinkage'],
     'samplingRate': gee_metrics[b]['params']['samplingRate'],
     'maxNodes': gee_metrics[b]['params']['maxNodes'],
     'RMSE_val': round(gee_metrics[b]['val_rmse'], 4)}
    for b in TARGETS
]).set_index('Banda')

display(df_best.style
    .background_gradient(cmap='RdYlGn_r', subset=['RMSE_val'])
    .set_caption('Melhores hiperparametros GBT por banda SWIR'))

In [ ]:
# ── RMSE por banda (barras) ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
rmse_vals = [gee_metrics[b]['val_rmse'] for b in TARGETS]
barras = ax.bar(TARGETS, rmse_vals, color='#E07B54', edgecolor='black', linewidth=0.5)
for bar, val in zip(barras, rmse_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Banda SWIR')
ax.set_ylabel('RMSE Validacao')
ax.set_title('RMSE GBT (GEE) — Melhores Params por Banda')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'gee_rmse_val.png', bbox_inches='tight')
plt.show()

## 5. Modelos Finais

Retreinamento de cada banda no **conjunto completo** (`fc_all`) com os melhores parâmetros.

In [ ]:
print('Treinando modelos finais (conjunto completo)...')
classifiers = {}

for banda in TARGETS:
    params = melhores_params[banda]
    classifiers[banda] = treinar_gbt(fc_all, banda, params)
    print(f'  {banda}  OK  '
          f'(nTrees={params["numberOfTrees"]}  lr={params["shrinkage"]}  '
          f'maxNodes={params["maxNodes"]})')

print('\nModelos finais prontos')

## 6. Aplicar ao Mosaico e Exportar

Cada classificador estima uma banda SWIR sobre o mosaico final a 30 m. O resultado é uma imagem com 6 bandas: `B04_est`, `B05_est`, ..., `B09_est`.

In [ ]:
mosaico      = ee.Image(ASSET_MOSAIC)
img_features = mosaico.select(FEATURES)
area         = mosaico.geometry()

# Aplicar cada classificador
bandas_est = [
    img_features.classify(classifiers[b], f'{b}_est')
    for b in TARGETS
]
img_estimada = (
    ee.Image.cat(bandas_est)
    .int16()
    .set('description', 'SWIR estimado via GBT (VNIR+TIR)')
)
print('Bandas estimadas:', img_estimada.bandNames().getInfo())

In [ ]:
# ── Mapa com bandas estimadas (requer geemap) ───────────────────────────────
if HAS_GEEMAP:
    mapa2 = geemap.Map()
    mapa2.centerObject(area, zoom=8)
    mapa2.addLayer(
        mosaico.divide(10000).float(),
        {'bands': ['B3N', 'B02', 'B01'], 'min': 0.05, 'max': 0.35, 'gamma': 1.4},
        'Mosaico RGB Original'
    )
    mapa2.addLayer(
        img_estimada.divide(10000).float(),
        {'bands': ['B04_est', 'B06_est', 'B3N'], 'min': 0.1, 'max': 0.5, 'gamma': 1.2},
        'SWIR Estimado Falsa-cor'
    )
    display(mapa2)
else:
    print('geemap nao disponivel para mapa interativo')

In [ ]:
# ── Exportar para asset GEE ────────────────────────────────────────────────
print(f'Exportando -> {ASSET_OUTPUT}')

task = ee.batch.Export.image.toAsset(
    image=img_estimada,
    description='swir_estimated_gbt',
    assetId=ASSET_OUTPUT,
    region=area,
    scale=30,
    maxPixels=1e13,
    pyramidingPolicy={'.default': 'mean'},
)
task.start()
print(f'Task iniciada: {task.id}')
print('Acompanhe em: https://code.earthengine.google.com/tasks')

In [ ]:
# ── Monitoramento opcional ────────────────────────────────────────────────
# Descomente para aguardar a task

# import time
# while True:
#     estado = task.status()['state']
#     print(f'  Status: {estado}')
#     if estado in ('COMPLETED', 'FAILED', 'CANCELLED'):
#         break
#     time.sleep(60)
# print('Task finalizada:', task.status()['state'])